In [1]:
# Load the toolbox and the data
import sys
sys.path.append("../src")

import math
import pandas as pd

from gow_rag.data_loader import load_corpus, load_claims
from gow_rag.preprocessing import make_tokenizer
from gow_rag.sparse_retrieval import SparseRetriever, count_words
from gow_rag.graph_of_words import graph_term_weights
from gow_rag.evaluation import evaluate
from gow_rag.paths import TABLES_DIR, DATA_DIR

corpus = load_corpus()
queries, qrels = load_claims("validation")
doc_ids = corpus["doc_id"].tolist()
print("Papers:", len(corpus), "| Claims:", len(queries))

Papers: 5183 | Claims: 188


## Preparing the text in 3 ways
- **A** is what we used so far.
- **B** removes the full standard English stopword list.
- **C** also stems words. This is the paper's preparation.

In [2]:
recipes = {
    "A: our stopwords, no stemming": make_tokenizer("project", stem=False),
    "B: full stopwords, no stemming": make_tokenizer("english", stem=False),
    "C: full stopwords + stemming (paper)": make_tokenizer("english", stem=True),
}

example = queries[list(queries)[0]]
print("Claim:", example)
for name, tokenizer in recipes.items():
    print()
    print(name)
    print("  ", tokenizer(example))

Claim: 1,000 genomes project enables mapping of genetic sequence variation consisting of rare variants with larger penetrance effects than common variants.

A: our stopwords, no stemming
   ['000', 'genomes', 'project', 'enables', 'mapping', 'genetic', 'sequence', 'variation', 'consisting', 'rare', 'variants', 'larger', 'penetrance', 'effects', 'than', 'common', 'variants']

B: full stopwords, no stemming
   ['000', 'genomes', 'project', 'enables', 'mapping', 'genetic', 'sequence', 'variation', 'consisting', 'rare', 'variants', 'larger', 'penetrance', 'effects', 'common', 'variants']

C: full stopwords + stemming (paper)
   ['000', 'genom', 'project', 'enabl', 'map', 'genet', 'sequenc', 'variat', 'consist', 'rare', 'variant', 'larger', 'penetr', 'effect', 'common', 'variant']


In [3]:
# scoring using bases we made
def run_experiment(tokenizer, graph_transform=None, twidf_b=None):
    """Build TF-IDF, BM25 and TW-IDF with one tokenizer and return their scores."""
    doc_tokens = [tokenizer(text) for text in corpus["full_text"]]
    doc_lengths = [len(tokens) for tokens in doc_tokens]
    doc_counts = [count_words(tokens) for tokens in doc_tokens]
    doc_graph = [graph_term_weights(tokens) for tokens in doc_tokens]
    if graph_transform is not None:
        doc_graph = [{w: graph_transform(v) for w, v in d.items() if v > 0} for d in doc_graph]

    engines = {
        "TF-IDF": SparseRetriever(doc_ids, doc_counts, doc_lengths, model="tfidf"),
        "BM25": SparseRetriever(doc_ids, doc_counts, doc_lengths, model="bm25"),
        "TW-IDF": SparseRetriever(doc_ids, doc_graph, doc_lengths, model="twidf", b=twidf_b),
    }
    return {name: evaluate(engine.search_all(queries, tokenizer), qrels)
            for name, engine in engines.items()}


all_scores = {name: run_experiment(tokenizer) for name, tokenizer in recipes.items()}

map_table = pd.DataFrame({recipe: {m: s["MAP"] for m, s in scores.items()}
                          for recipe, scores in all_scores.items()}).T
map_table.index.name = "Text preparation (MAP shown)"
map_table

,TF-IDF,BM25,TW-IDF
Text preparation (MAP shown),,,
"A: our stopwords, no stemming",0.7601,0.7704,0.5999
"B: full stopwords, no stemming",0.7638,0.7707,0.6030
C: full stopwords + stemming (paper),0.7822,0.7809,0.6036


In [4]:
paper_tokenizer = recipes["C: full stopwords + stemming (paper)"]
query_lengths = [len(paper_tokenizer(text)) for text in queries.values()]
print("Average claim length:", round(sum(query_lengths) / len(query_lengths), 1), "words")
print("Shortest:", min(query_lengths), "| Longest:", max(query_lengths))

Average claim length: 8.7 words
Shortest: 3 | Longest: 19


In [5]:
damp = lambda tw: 1.0 + math.log(1.0 + math.log(tw))

variants = {
    "TW-IDF, b=0.003 (paper)": run_experiment(paper_tokenizer)["TW-IDF"],
    "TW-IDF, b=0.20": run_experiment(paper_tokenizer, twidf_b=0.20)["TW-IDF"],
    "TW-IDF, b=0.75": run_experiment(paper_tokenizer, twidf_b=0.75)["TW-IDF"],
    "TW-IDF damped, b=0.003": run_experiment(paper_tokenizer, graph_transform=damp)["TW-IDF"],
    "TW-IDF damped, b=0.20": run_experiment(paper_tokenizer, graph_transform=damp, twidf_b=0.20)["TW-IDF"],
    "BM25 (for reference)": all_scores["C: full stopwords + stemming (paper)"]["BM25"],
}
variant_table = pd.DataFrame(variants).T
variant_table.index.name = "Method"
variant_table

,MAP,P@10,Recall@10,MRR@10,nDCG@10
Method,,,,,
"TW-IDF, b=0.003 (paper)",0.6036,0.0883,0.8335,0.6069,0.6580
"TW-IDF, b=0.20",0.6303,0.0910,0.8566,0.6337,0.6841
"TW-IDF, b=0.75",0.6438,0.0941,0.8942,0.6510,0.7046
"TW-IDF damped, b=0.003",0.7574,0.0973,0.9098,0.7601,0.7939
"TW-IDF damped, b=0.20",0.7552,0.0989,0.9222,0.7597,0.7963
BM25 (for reference),0.7809,0.0995,0.9235,0.7836,0.8159


In [6]:
#saving the table
map_table.to_csv(TABLES_DIR / "diagnosis_text_preparation_map.csv")
variant_table.to_csv(TABLES_DIR / "diagnosis_twidf_variants.csv")
print("Saved 2 tables in:", TABLES_DIR)

Saved 2 tables in: C:\Users\Lenovo\Desktop\Projects for IP Paris Application\Graph of words (Reviewing the paper with dataset)\github_upload\scientific-literature-GOW-RAG\results\tables


In [7]:
#local copy for data in system
from datasets import load_dataset

load_dataset("allenai/scifact", "corpus")["train"].to_pandas().to_json(
    DATA_DIR / "scifact_corpus.jsonl", orient="records", lines=True)
for split in ["train", "validation"]:
    load_dataset("allenai/scifact", "claims")[split].to_pandas().to_json(
        DATA_DIR / f"scifact_claims_{split}.jsonl", orient="records", lines=True)
print("Saved 3 data files in:", DATA_DIR)

Saved 3 data files in: C:\Users\Lenovo\Desktop\Projects for IP Paris Application\Graph of words (Reviewing the paper with dataset)\github_upload\scientific-literature-GOW-RAG\data
